# 🗺️ Txoko — Enriquecimiento con Google Maps Places API
### Reto Inetum · Bootcamp BBK The Bridge · Equipo 4  
**Objetivo:** Añadir `google_rating`, `google_num_reviews` y `google_place_id` al `txoko_master.csv`  
**Fuente:** Google Maps Places API (oficial, gratuita hasta $200/mes de crédito)  
**Licencia datos**: Google Maps Platform Terms of Service — solo usamos rating + num_reviews

---

## 🔑 PASO 0 — Obtener tu Google Maps API Key (5 minutos, gratis)

Sigue estos pasos **antes de ejecutar el notebook**:

1. Ve a [https://console.cloud.google.com](https://console.cloud.google.com) e inicia sesión con tu cuenta Google  
2. Crea un proyecto nuevo (ej: `txoko-demo`) o usa uno existente  
3. En el menú izquierdo → **APIs y servicios** → **Biblioteca**  
4. Busca **"Places API"** → haz clic → **Habilitar**  
5. Ve a **APIs y servicios** → **Credenciales** → **Crear credenciales** → **Clave de API**  
6. Copia la clave que aparece (formato: `AIzaSy...`)  
7. **Opcional pero recomendado**: restringe la clave a "Places API" para mayor seguridad  
8. Google te da **$200 de crédito gratuito al mes** — más que suficiente para los 4.624 lugares

> ⚠️ **Para el jurado**: la clave de API es credencial personal. En el notebook solo aparece como variable de entorno, nunca hardcodeada.


In [ ]:
# Instalar dependencias (ejecutar solo si es necesario)
# En Google Colab ya están disponibles; en local descomentar si falta alguna
# !pip install requests pandas numpy tqdm
print("✓ Librerías listas")

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import json
import os
from math import radians, sin, cos, sqrt, atan2
from tqdm.notebook import tqdm   # barra de progreso en Jupyter

print(f"✓ pandas {pd.__version__} | numpy {np.__version__}")
print("✓ Todas las librerías importadas")

In [ ]:
# ─── CONFIGURACIÓN — pon aquí tu clave de Google Maps ──────────────────────
#
# OPCIÓN 1 (más segura): clave como variable de entorno
#   En terminal: export GOOGLE_MAPS_API_KEY='AIzaSy...'
#
# OPCIÓN 2 (para Colab o demo rápida): pega la clave directamente
#   Descomenta la línea de abajo y pon tu clave entre comillas

# GOOGLE_MAPS_API_KEY = "AIzaSy_PEGA_TU_CLAVE_AQUI"   # ← descomentar si no usas variable de entorno

GOOGLE_MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY", "")

if not GOOGLE_MAPS_API_KEY:
    print("⚠️  API Key no configurada.")
    print("   Opción 1: En terminal → export GOOGLE_MAPS_API_KEY='tu-clave'")
    print("   Opción 2: Descomenta la línea de arriba y pega tu clave")
else:
    # Mostrar solo los primeros y últimos caracteres por seguridad
    masked = GOOGLE_MAPS_API_KEY[:8] + "..." + GOOGLE_MAPS_API_KEY[-4:]
    print(f"✓ API Key cargada: {masked}")

In [ ]:
# ─── PARÁMETROS DEL PIPELINE ─────────────────────────────────────────────────

# Ruta al CSV maestro (ajusta si es necesario)
INPUT_CSV  = "txoko_master.csv"
OUTPUT_CSV = "txoko_master_enriched.csv"
CACHE_FILE = "google_reviews_cache.json"   # evita repetir peticiones si el notebook se interrumpe

# Límite para la demo (None = procesar todos los 4.624 lugares)
# Con 200 estamos dentro del tier gratuito de Google ($200 crédito = ~11.700 peticiones)
DEMO_LIMIT = 200

# Umbrales de distancia para validar que el resultado de Google es el lugar correcto
MATCH_THRESHOLD_HIGH   = 300   # < 300m  → confianza ALTA  ✅
MATCH_THRESHOLD_MEDIUM = 800   # < 800m  → confianza MEDIA ✅
# > 800m → confianza BAJA ❌ (no se incluye en el dataset)

# Pausa entre peticiones (segundos) — cortesía a la API de Google
SLEEP_BETWEEN_REQUESTS = 0.3

print("✓ Configuración lista")
print(f"  Input : {INPUT_CSV}")
print(f"  Output: {OUTPUT_CSV}")
print(f"  Demo  : {DEMO_LIMIT} lugares" if DEMO_LIMIT else "  Modo  : COMPLETO (4.624 lugares)")

In [ ]:
# ─── CARGAR DATASET MAESTRO ──────────────────────────────────────────────────

df = pd.read_csv(INPUT_CSV)
print(f"✓ Dataset cargado: {len(df):,} lugares × {len(df.columns)} columnas")
print(f"  Con coordenadas: {df['lat'].notna().sum():,} ({df['lat'].notna().mean()*100:.1f}%)")
print(f"\nCategorías:")
print(df['categoria'].value_counts().to_string())
df.head(3)

In [ ]:
# ─── SELECCIÓN DE MUESTRA PARA LA DEMO ───────────────────────────────────────
#
# Estrategia: representación balanceada por categoría,
# priorizando lugares CON coordenadas (más fácil validar el match geográfico)

if DEMO_LIMIT:
    df_with_coords = df[df['lat'].notna()].copy()
    n_per_cat = max(1, DEMO_LIMIT // df['categoria'].nunique())
    
    df_sample = (df_with_coords
                 .groupby('categoria', group_keys=False)
                 .apply(lambda x: x.head(n_per_cat))
                 .head(DEMO_LIMIT)
                 .copy())
    
    print(f"✓ Muestra seleccionada: {len(df_sample)} lugares")
    print(f"  Distribución por categoría:")
    print(df_sample['categoria'].value_counts().to_string())
else:
    df_sample = df.copy()
    print(f"✓ Procesando dataset completo: {len(df_sample):,} lugares")

In [ ]:
# ─── FUNCIONES AUXILIARES ─────────────────────────────────────────────────────

def haversine_meters(lat1, lon1, lat2, lon2):
    """
    Distancia en metros entre dos puntos GPS usando la fórmula haversine.
    
    Por qué la usamos: cuando Google Maps nos devuelve un resultado para
    'RESTAURANTE ELKANO Getaria', verificamos que las coordenadas del resultado
    estén a menos de 300m de las coordenadas que tenemos en el dataset.
    Si está a más distancia, probablemente es un lugar con nombre similar pero distinto.
    """
    R = 6371000  # radio de la Tierra en metros
    phi1, phi2 = radians(lat1), radians(lat2)
    dphi = radians(lat2 - lat1)
    dlam = radians(lon2 - lon1)
    a = sin(dphi/2)**2 + cos(phi1)*cos(phi2)*sin(dlam/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))


def load_cache(path):
    """Carga el caché de peticiones previas para no repetir llamadas a la API."""
    if os.path.exists(path):
        with open(path, encoding='utf-8') as f:
            return json.load(f)
    return {}


def save_cache(path, cache):
    """Guarda el caché a disco."""
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

print("✓ Funciones auxiliares definidas (haversine, caché)")

# Test rápido de haversine — Bilbao a San Sebastián ≈ 100km
dist = haversine_meters(43.2630, -2.9350, 43.3183, -1.9812)
print(f"  Test: Bilbao → Donostia = {dist/1000:.1f} km (esperado ~100 km) ✓")

In [ ]:
# ─── FUNCIÓN DE BÚSQUEDA EN GOOGLE MAPS PLACES API ───────────────────────────
#
# Documentación oficial:
#   https://developers.google.com/maps/documentation/places/web-service/text-search
#
# Coste: ~$0.017 por petición (Text Search)
# Con $200 de crédito gratuito mensual → ~11.700 peticiones disponibles
# Nuestros 4.624 lugares → ~$79 → dentro del crédito gratuito

def google_places_lookup(nombre, municipio, lat=None, lon=None):
    """
    Busca un lugar en Google Maps Places API (Text Search).
    
    Args:
        nombre   : nombre del lugar (ej: 'RESTAURANTE ELKANO')
        municipio: municipio (ej: 'Getaria')
        lat, lon : coordenadas del CSV para validar el match
    
    Returns:
        dict con place_id, rating, num_reviews, result_lat, result_lon
        o None si no se encontró resultado
    """
    query = f"{nombre} {municipio} Euskadi"
    url = "https://maps.googleapis.com/maps/api/place/textsearch/json"
    params = {
        "query"   : query,
        "key"     : GOOGLE_MAPS_API_KEY,
        "language": "es",
    }
    # Si tenemos coords del CSV, añadir location bias (mejora mucho la precisión)
    if lat and lon:
        params["location"] = f"{lat},{lon}"
        params["radius"]   = 5000  # buscar en radio de 5km desde el lugar conocido

    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    
    # Google devuelve status OK si hay resultados, ZERO_RESULTS si no
    if data.get("status") != "OK" or not data.get("results"):
        return None
    
    r = data["results"][0]
    return {
        "place_id"  : r.get("place_id"),
        "rating"    : r.get("rating"),
        "num_reviews": r.get("user_ratings_total"),
        "result_lat": r["geometry"]["location"]["lat"],
        "result_lon": r["geometry"]["location"]["lng"],
    }


def enrich_place(nombre, municipio, lat, lon, cache):
    """
    Busca el lugar en Google Maps, valida el match por distancia haversine,
    y devuelve los datos enriquecidos con nivel de confianza.
    """
    cache_key = f"{nombre}||{municipio}"
    
    # Usar caché si ya buscamos este lugar antes
    if cache_key in cache:
        return cache[cache_key]
    
    try:
        result = google_places_lookup(nombre, municipio, lat, lon)
        
        if result is None:
            out = {
                "google_place_id"   : None,
                "google_rating"     : None,
                "google_num_reviews": None,
                "google_match_conf" : "no_match",
            }
        else:
            # ── Validar match por distancia ────────────────────────────────
            conf = "low"
            if (lat and lon and 
                result.get("result_lat") and result.get("result_lon")):
                dist = haversine_meters(
                    lat, lon,
                    result["result_lat"], result["result_lon"]
                )
                if   dist < MATCH_THRESHOLD_HIGH  : conf = "high"
                elif dist < MATCH_THRESHOLD_MEDIUM: conf = "medium"
                # else conf = "low" → no usamos este match
            else:
                # Sin coords para validar: asumimos medium si hay rating
                conf = "medium" if result.get("rating") else "low"
            
            out = {
                "google_place_id"   : result.get("place_id"),
                "google_rating"     : result.get("rating")      if conf in ("high","medium") else None,
                "google_num_reviews": result.get("num_reviews") if conf in ("high","medium") else None,
                "google_match_conf" : conf,
            }
    
    except Exception as e:
        out = {
            "google_place_id"   : None,
            "google_rating"     : None,
            "google_num_reviews": None,
            "google_match_conf" : "error",
        }
        print(f"  ⚠ Error en '{nombre}': {e}")
    
    cache[cache_key] = out
    return out

print("✓ Funciones de búsqueda y validación definidas")

In [ ]:
# ─── EJECUTAR EL PIPELINE DE ENRIQUECIMIENTO ─────────────────────────────────
#
# Esta celda hace las peticiones a Google Maps.
# Si la interrumpes a mitad, el caché guarda el progreso — puedes reejecutar
# sin repetir las peticiones ya realizadas.

# Verificar que la API key está configurada
if not GOOGLE_MAPS_API_KEY:
    raise ValueError(
        "⛔ API Key no configurada. "
        "Vuelve a la celda de configuración y sigue las instrucciones."
    )

# Cargar caché de ejecuciones anteriores
cache = load_cache(CACHE_FILE)
print(f"✓ Caché cargada: {len(cache)} entradas previas")
print(f"  Lugares a procesar: {len(df_sample)}")
print(f"  Nuevas peticiones necesarias: {len(df_sample) - sum(1 for r in df_sample.itertuples() if f'{r.nombre}||{r.municipio}' in cache)}")
print()

results = []

for row in tqdm(df_sample.itertuples(), total=len(df_sample), desc="Google Maps lookup"):
    enrich = enrich_place(
        nombre    = row.nombre,
        municipio = row.municipio,
        lat       = row.lat if pd.notna(row.lat) else None,
        lon       = row.lon if pd.notna(row.lon) else None,
        cache     = cache,
    )
    results.append(enrich)
    
    # Guardar caché cada 50 peticiones
    if len(results) % 50 == 0:
        save_cache(CACHE_FILE, cache)
    
    time.sleep(SLEEP_BETWEEN_REQUESTS)

# Guardar caché final
save_cache(CACHE_FILE, cache)
print(f"\n✓ Pipeline completado | Caché guardada en {CACHE_FILE}")

In [ ]:
# ─── MERGE CON EL DATASET MAESTRO ────────────────────────────────────────────

df_enrich = pd.DataFrame(results, index=df_sample.index)
df_out    = df_sample.join(df_enrich)

# ── Estadísticas de calidad del enriquecimiento ───────────────────────────────
matched_high   = (df_out['google_match_conf'] == 'high').sum()
matched_medium = (df_out['google_match_conf'] == 'medium').sum()
no_match       = df_out['google_match_conf'].isin(['low','no_match','error']).sum()
has_rating     = df_out['google_rating'].notna().sum()

print(f"{'='*55}")
print(f"RESULTADO DEL ENRIQUECIMIENTO — {len(df_out)} lugares")
print(f"{'='*55}")
print(f"  Match HIGH   (dist < 300m) : {matched_high:4d}  ({matched_high/len(df_out)*100:.1f}%)")
print(f"  Match MEDIUM (dist < 800m) : {matched_medium:4d}  ({matched_medium/len(df_out)*100:.1f}%)")
print(f"  Sin match válido           : {no_match:4d}  ({no_match/len(df_out)*100:.1f}%)")
print(f"  Con google_rating          : {has_rating:4d}  ({has_rating/len(df_out)*100:.1f}%)")

if has_rating > 0:
    avg = df_out['google_rating'].dropna().mean()
    print(f"  Rating medio               : {avg:.2f} / 5.0")

print()
df_out[['id','nombre','municipio','categoria',
        'google_rating','google_num_reviews','google_match_conf']].head(10)

In [ ]:
# ─── GUARDAR DATASET ENRIQUECIDO ─────────────────────────────────────────────

df_out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

size_kb = os.path.getsize(OUTPUT_CSV) // 1024
print(f"✓ Dataset guardado: {OUTPUT_CSV}")
print(f"  {len(df_out):,} registros × {len(df_out.columns)} columnas  ({size_kb} KB)")
print()
print("Columnas nuevas añadidas:")
for col in ['google_place_id','google_rating','google_num_reviews','google_match_conf']:
    filled = df_out[col].notna().sum()
    print(f"  {col:25s}: {filled}/{len(df_out)} rellenos")

In [ ]:
# ─── ANÁLISIS PARA EL JURADO ─────────────────────────────────────────────────
# Top lugares por rating (solo matches de alta confianza)

df_high = df_out[df_out['google_match_conf'].isin(['high','medium'])].copy()

print("TOP 15 lugares por Google Rating (alta/media confianza):")
print()
top = (df_high[df_high['google_rating'].notna()]
       .sort_values(['google_rating','google_num_reviews'], ascending=[False,False])
       [['nombre','municipio','categoria','google_rating','google_num_reviews']]
       .head(15))
print(top.to_string(index=False))

print()
print("Rating medio por categoría:")
cat_rating = (df_high.groupby('categoria')['google_rating']
              .agg(['mean','count'])
              .rename(columns={'mean':'rating_medio','count':'n_con_rating'})
              .sort_values('rating_medio', ascending=False))
print(cat_rating.to_string())

In [ ]:
# ─── TRAZABILIDAD PARA EL JURADO ─────────────────────────────────────────────
# Documentar el origen de cada campo — fundamental para la presentación

provenance = {
    "dataset": "txoko_master_enriched.csv",
    "fecha_enriquecimiento": pd.Timestamp.now().isoformat(),
    "campos_opendata_euskadi": [
        "id", "source_dataset", "source_url_json", "categoria", "subcategoria",
        "nombre", "descripcion", "municipio", "territorio",
        "lat", "lon", "direccion", "telefono", "web", "ficha_turismo"
    ],
    "campos_google_maps": [
        "google_place_id", "google_rating", "google_num_reviews", "google_match_conf"
    ],
    "fuente_google": {
        "api": "Google Maps Places API — Text Search",
        "url_docs": "https://developers.google.com/maps/documentation/places/web-service/text-search",
        "licencia": "Google Maps Platform Terms of Service",
        "uso_permitido": "rating y num_reviews — solo lectura, no redistribución de reseñas completas",
        "coste": "gratuito hasta $200/mes de crédito Google Cloud"
    },
    "estrategia_matching": {
        "metodo": "Text Search por nombre + municipio + 'Euskadi'",
        "validacion": "distancia haversine entre coords CSV y coords Google Maps",
        "umbral_high": f"< {MATCH_THRESHOLD_HIGH}m",
        "umbral_medium": f"< {MATCH_THRESHOLD_MEDIUM}m",
        "descartado": f"> {MATCH_THRESHOLD_MEDIUM}m"
    },
    "stats": {
        "lugares_procesados": int(len(df_out)),
        "con_rating": int(df_out['google_rating'].notna().sum()),
        "match_high": int((df_out['google_match_conf']=='high').sum()),
        "match_medium": int((df_out['google_match_conf']=='medium').sum()),
    }
}

with open("txoko_enrichment_provenance.json", "w", encoding="utf-8") as f:
    json.dump(provenance, f, ensure_ascii=False, indent=2)

print("✓ Trazabilidad guardada: txoko_enrichment_provenance.json")
print()
print(json.dumps(provenance, ensure_ascii=False, indent=2))